In [1]:
!pip install mediapipe==0.10.5 --upgrade --force-reinstall


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 3.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.9/107.9 kB 8.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.5/33.5 MB 27.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.8/63.8 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.1/162.1 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 108.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.2/73.2 MB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.8/16.8 MB 96.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 467.2/467.2 kB 27.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 355.2/355.2 kB 21.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# Crunch: ZIP -> Extract -> FAST keypoints -> Rule labels(정자세=1) -> CSV/PKL
# ===================================================================

# 0) 경로만 맞춰줘
ZIP_PATH   = "/content/drive/MyDrive/크런치.zip"          # 드라이브에 올린 ZIP
IMG_ROOT   = "/content/drive/MyDrive/crunch/images"       # 압축 풀릴 폴더
OUTPUT_DIR = "/content/drive/MyDrive/crunch/outputs"      # 결과 저장 폴더
CSV_OUT    = f"{OUTPUT_DIR}/crunch_labels.csv"
PKL_OUT    = f"{OUTPUT_DIR}/crunch_dataset.pkl"

import os, glob, math, zipfile, pickle, warnings
import numpy as np, pandas as pd
from tqdm.auto import tqdm
import cv2, mediapipe as mp
warnings.filterwarnings("ignore")

os.makedirs(IMG_ROOT, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 1) ZIP 추출 (이미 풀려있으면 건너뜀)
def safe_extract_zip(zip_path, out_dir):
    if not os.path.exists(zip_path):
        print(f"[WARN] ZIP 없음: {zip_path}")
        return
    has_files = any(True for _ in glob.iglob(os.path.join(out_dir, "**", "*"), recursive=True))
    if has_files:
        print("[INFO] IMG_ROOT에 파일 존재 → 추출 생략")
        return
    print(f"[INFO] 압축 해제: {zip_path} -> {out_dir}")
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(out_dir)

safe_extract_zip(ZIP_PATH, IMG_ROOT)

# 2) MediaPipe 좌표 추출 유틸
mp_pose = mp.solutions.pose
LANDMARKS = [
    'nose','left_eye_inner','left_eye','left_eye_outer','right_eye_inner','right_eye','right_eye_outer',
    'left_ear','right_ear','mouth_left','mouth_right',
    'left_shoulder','right_shoulder','left_elbow','right_elbow','left_wrist','right_wrist',
    'left_pinky','right_pinky','left_index','right_index','left_thumb','right_thumb',
    'left_hip','right_hip','left_knee','right_knee','left_ankle','right_ankle',
    'left_heel','right_heel','left_foot_index','right_foot_index'
]

def angle_3pt(a, b, c):
    try:
        ab = (a[0]-b[0], a[1]-b[1]); cb = (c[0]-b[0], c[1]-b[1])
        dot = ab[0]*cb[0] + ab[1]*cb[1]
        nab = math.hypot(*ab); ncb = math.hypot(*cb)
        cosang = max(-1., min(1., dot/((nab*ncb)+1e-9)))
        return math.degrees(math.acos(cosang))
    except:
        return float("nan")

def dist(a,b): return float(math.hypot(a[0]-b[0], a[1]-b[1]))
def mid(p,q):  return ((p[0]+q[0])/2.0, (p[1]+q[1])/2.0)

# i3 최적화: 긴 변 256 리사이즈
def fast_resize(img, max_side=256):
    h, w = img.shape[:2]
    if max(h, w) <= max_side: return img
    if h >= w:
        nh = max_side; nw = int(w * (max_side / h))
    else:
        nw = max_side; nh = int(h * (max_side / w))
    return cv2.resize(img, (nw, nh), interpolation=cv2.INTER_AREA)

def extract_keypoints(img_bgr):
    img_s = fast_resize(img_bgr, 256)
    with mp_pose.Pose(static_image_mode=True, model_complexity=1,
                      enable_segmentation=False, min_detection_confidence=0.5) as pose:
        res = pose.process(cv2.cvtColor(img_s, cv2.COLOR_BGR2RGB))
        if not res.pose_landmarks:
            return None, img_s.shape[:2]
        lm = res.pose_landmarks.landmark
        H, W = img_s.shape[:2]
        kps = np.array([[p.x*W, p.y*H, p.visibility] for p in lm], dtype=np.float32)  # (33,3)
        return kps, (H, W)

# 3) 크런치 정자세 규칙 (정자세=1, 오자세=0)
# 핵심 의도(이미지 기준, y는 아래로 증가):
# - Torso flexion 충분: hip에서 본 (어깨-엉덩이-무릎) 각도 평균 ≤ 120°
# - 어깨가 무릎 쪽으로 충분히 접근: dist(어깨,무릎)/dist(엉덩이,무릎) 평균 ≤ 0.85
# - 골반/몸통 과기울기 없음: |어깨x-엉덩이x|/dist(어깨,엉덩이) 평균 ≤ 0.35
# - 목 중립(느슨): 귀-어깨-엉덩이 각도 110°~170° (한쪽만 만족해도 OK)
CRUNCH_THRESH = {
    "hip_angle_max": 120.0,
    "shoulder_knee_ratio": 0.85,
    "pelvis_stability": 0.35,
    "neck_min": 110.0,
    "neck_max": 170.0
}

def label_crunch(kps, hw):
    H, W = hw
    idx = {n:i for i,n in enumerate(LANDMARKS)}
    def P(name):
        j = idx[name]; return (float(kps[j,0]), float(kps[j,1]))

    LS, RS = P('left_shoulder'), P('right_shoulder')
    LH, RH = P('left_hip'), P('right_hip')
    LK, RK = P('left_knee'), P('right_knee')
    LEar, REar = P('left_ear'), P('right_ear')

    # hip angle (어깨-엉덩이-무릎)
    hip_ang_l = angle_3pt(LS, LH, LK)
    hip_ang_r = angle_3pt(RS, RH, RK)
    hip_ang   = np.nanmean([hip_ang_l, hip_ang_r])
    torso_flex_ok = hip_ang <= CRUNCH_THRESH["hip_angle_max"]

    # 어깨-무릎 접근 비율
    sk_l = dist(LS, LK) / (dist(LH, LK) + 1e-6)
    sk_r = dist(RS, RK) / (dist(RH, RK) + 1e-6)
    sk_ratio = np.nanmean([sk_l, sk_r])
    shoulder_to_knee_ok = sk_ratio <= CRUNCH_THRESH["shoulder_knee_ratio"]

    # 골반/몸통 기울기 (수평 이동 비율)
    torso_dx_l = abs(LS[0] - LH[0]) / (dist(LS, LH) + 1e-6)
    torso_dx_r = abs(RS[0] - RH[0]) / (dist(RS, RH) + 1e-6)
    pelvis_ok  = ((torso_dx_l + torso_dx_r)/2.0) <= CRUNCH_THRESH["pelvis_stability"]

    # 목 중립
    neck_l = angle_3pt(LEar, LS, LH)
    neck_r = angle_3pt(REar, RS, RH)
    neck_ok = (CRUNCH_THRESH["neck_min"] <= neck_l <= CRUNCH_THRESH["neck_max"]) or \
              (CRUNCH_THRESH["neck_min"] <= neck_r <= CRUNCH_THRESH["neck_max"])

    label = 1 if (sum([torso_flex_ok, shoulder_to_knee_ok, pelvis_ok]) >= 2 and neck_ok) else 0
    flags = {
        "hip_angle": float(hip_ang),
        "sk_ratio": float(sk_ratio),
        "torso_dx_l": float(torso_dx_l),
        "torso_dx_r": float(torso_dx_r),
        "torso_flex_ok": int(torso_flex_ok),
        "shoulder_to_knee_ok": int(shoulder_to_knee_ok),
        "pelvis_ok": int(pelvis_ok),
        "neck_ok": int(neck_ok)
    }
    return label, flags

# 4) 이미지 수집
def image_paths_from(root):
    exts = ("*.jpg","*.jpeg","*.png","*.bmp","*.webp")
    files = []
    for e in exts:
        files.extend(glob.glob(os.path.join(root, "**", e), recursive=True))
    return sorted(list(set(files)))

all_imgs = image_paths_from(IMG_ROOT)
print(f"[INFO] 이미지 개수: {len(all_imgs)}")

# 5) 실행 → CSV/PKL 저장
rows = []
for p in tqdm(all_imgs, desc="Crunch labeling"):
    img = cv2.imread(p)
    if img is None:
        continue
    kps, hw = extract_keypoints(img)
    if kps is None:
        continue
    y, f = label_crunch(kps, hw)
    rows.append({
        "path": p, "label": int(y),
        "hip_angle": f["hip_angle"], "sk_ratio": f["sk_ratio"],
        "torso_dx_l": f["torso_dx_l"], "torso_dx_r": f["torso_dx_r"],
        "torso_flex_ok": f["torso_flex_ok"], "shoulder_to_knee_ok": f["shoulder_to_knee_ok"],
        "pelvis_ok": f["pelvis_ok"], "neck_ok": f["neck_ok"]
    })

df = pd.DataFrame(rows)
df.to_csv(CSV_OUT, index=False)

# 모델 학습용 간단 피처 (4개)
X_cols = ["hip_angle","sk_ratio","torso_dx_l","torso_dx_r"]
X = df[X_cols].fillna(0.0).to_numpy(dtype=np.float32)
y = df["label"].fillna(0).to_numpy(dtype=np.int64)
with open(PKL_OUT, "wb") as f:
    pickle.dump({"X": X, "y": y, "feature_cols": X_cols}, f)

print(f"[DONE] 저장 완료\n- CSV: {CSV_OUT}\n- PKL: {PKL_OUT}\n총 샘플: {len(df)}")


[INFO] 압축 해제: /content/drive/MyDrive/크런치.zip -> /content/drive/MyDrive/crunch/images
[INFO] 이미지 개수: 3349


Crunch labeling:   0%|          | 0/3349 [00:00<?, ?it/s]

[DONE] 저장 완료
- CSV: /content/drive/MyDrive/crunch/outputs/crunch_labels.csv
- PKL: /content/drive/MyDrive/crunch/outputs/crunch_dataset.pkl
총 샘플: 3237


In [3]:
# Crunch 모델 학습 (PKL 우선, 없으면 CSV 사용)
# 산출물: model.pkl, feature_cols.json, metrics.json, counts.json, pred_val.csv
# -----------------------------------------------------------------------------

# 0) 경로 (필요 시만 수정)
BASE       = OUTPUT_DIR  # 이전 셀에서 만든 경로 재사용
DATA_PKL   = f"{BASE}/crunch_dataset.pkl"
DATA_CSV   = f"{BASE}/crunch_labels.csv"

import os, json, pickle, warnings
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score, confusion_matrix

# 1) 데이터 로드 (PKL 우선)
if os.path.exists(DATA_PKL):
    with open(DATA_PKL, "rb") as f:
        blob = pickle.load(f)
    X = np.asarray(blob["X"], dtype=np.float32)
    y = np.asarray(blob["y"], dtype=np.int64)
    feature_cols = blob.get("feature_cols")
    print(f"[LOAD] PKL -> X:{X.shape}, y:{y.shape}, cols={feature_cols}")
elif os.path.exists(DATA_CSV):
    df = pd.read_csv(DATA_CSV)
    default_cols = ["hip_angle","sk_ratio","torso_dx_l","torso_dx_r"]
    feature_cols = [c for c in default_cols if c in df.columns]
    X = df[feature_cols].fillna(0.0).to_numpy(np.float32)
    y = df["label"].fillna(0).astype("int64").to_numpy()
    print(f"[LOAD] CSV -> X:{X.shape}, y:{y.shape}, cols={feature_cols}")
else:
    raise FileNotFoundError("라벨링 파일을 찾을 수 없습니다. BASE/PKL/CSV 경로 확인!")

# 2) 정/오자세 개수 출력·저장
def cnt(lbl):
    c = Counter(lbl); return {"good_1": int(c.get(1,0)), "bad_0": int(c.get(0,0)), "total": int(len(lbl))}
counts = {"all": cnt(y)}
print(f"[COUNT ALL] 정자세(1):{counts['all']['good_1']} | 오자세(0):{counts['all']['bad_0']} | 전체:{counts['all']['total']}")

# 3) Train/Val 분할
X_tr, X_va, y_tr, y_va = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
counts["train"] = cnt(y_tr); counts["val"] = cnt(y_va)
print("[COUNT TRAIN]", counts["train"])
print("[COUNT VAL]  ", counts["val"])

# 4) 가벼운 모델 2개 (LogReg / RandomForest)
pipe_lr = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=2000, class_weight="balanced"))
])
rf = RandomForestClassifier(
    n_estimators=200, max_depth=6, min_samples_leaf=3,
    class_weight="balanced", n_jobs=-1, random_state=42
)

pipe_lr.fit(X_tr, y_tr)
rf.fit(X_tr, y_tr)

def eval_m(m, Xv, yv, name):
    prob = m.predict_proba(Xv)[:,1] if hasattr(m,"predict_proba") else m.decision_function(Xv)
    pred = (prob >= 0.5).astype(int)
    return {
        "name": name,
        "acc": float(accuracy_score(yv, pred)),
        "f1": float(f1_score(yv, pred, zero_division=0)),
        "precision": float(precision_score(yv, pred, zero_division=0)),
        "recall": float(recall_score(yv, pred, zero_division=0)),
        "roc_auc": float(roc_auc_score(yv, prob)),
        "cm": confusion_matrix(yv, pred).tolist()
    }, prob

m_lr, prob_lr = eval_m(pipe_lr, X_va, y_va, "LogReg")
m_rf, prob_rf = eval_m(rf,      X_va, y_va, "RandomForest")
best_name, best_model, best_metrics, best_prob = ("LogReg", pipe_lr, m_lr, prob_lr)
if m_rf["roc_auc"] > m_lr["roc_auc"]:
    best_name, best_model, best_metrics, best_prob = ("RandomForest", rf, m_rf, prob_rf)

print("[RESULT] LogReg:", m_lr)
print("[RESULT] RandomForest:", m_rf)
print("[SELECT] Best:", best_name)

# 5) 최적 threshold(F1 최대)
best_thr, best_f1 = 0.5, -1.0
for t in np.linspace(0.1, 0.9, 33):
    f1 = f1_score(y_va, (best_prob >= t).astype(int), zero_division=0)
    if f1 > best_f1:
        best_f1, best_thr = float(f1), float(t)

# 6) 산출물 저장
with open(f"{BASE}/model.pkl","wb") as f:
    pickle.dump({"model":best_model,"feature_cols":feature_cols,"threshold":best_thr,
                 "meta":{"type":"crunch_binary","selected_model":best_name}}, f)
with open(f"{BASE}/feature_cols.json","w",encoding="utf-8") as f:
    json.dump(feature_cols, f, ensure_ascii=False, indent=2)
with open(f"{BASE}/metrics.json","w",encoding="utf-8") as f:
    json.dump({"LogReg":m_lr,"RandomForest":m_rf,"selected":best_name,
               "best_threshold":best_thr,"best_val_f1":best_f1}, f, ensure_ascii=False, indent=2)
with open(f"{BASE}/counts.json","w",encoding="utf-8") as f:
    json.dump(counts, f, ensure_ascii=False, indent=2)

pd.DataFrame({
    "y_true": y_va,
    "prob_good": best_prob,
    "pred@0.50": (best_prob>=0.50).astype(int),
    f"pred@{best_thr:.2f}": (best_prob>=best_thr).astype(int)
}).to_csv(f"{BASE}/pred_val.csv", index=False)

print(f"[DONE] saved -> {BASE}")


[LOAD] PKL -> X:(3237, 4), y:(3237,), cols=['hip_angle', 'sk_ratio', 'torso_dx_l', 'torso_dx_r']
[COUNT ALL] 정자세(1):42 | 오자세(0):3195 | 전체:3237
[COUNT TRAIN] {'good_1': 34, 'bad_0': 2555, 'total': 2589}
[COUNT VAL]   {'good_1': 8, 'bad_0': 640, 'total': 648}
[RESULT] LogReg: {'name': 'LogReg', 'acc': 0.9814814814814815, 'f1': 0.5714285714285714, 'precision': 0.4, 'recall': 1.0, 'roc_auc': 0.9982421875, 'cm': [[628, 12], [0, 8]]}
[RESULT] RandomForest: {'name': 'RandomForest', 'acc': 0.9953703703703703, 'f1': 0.8235294117647058, 'precision': 0.7777777777777778, 'recall': 0.875, 'roc_auc': 0.998828125, 'cm': [[638, 2], [1, 7]]}
[SELECT] Best: RandomForest
[DONE] saved -> /content/drive/MyDrive/crunch/outputs


In [4]:
# Crunch 재학습: 소수 클래스 오버샘플 + 피처 증강 -> 모델 v2 저장
# 산출물: model_v2.pkl, feature_cols_v2.json, metrics_v2.json, counts_v2.json, pred_val_v2.csv
# ------------------------------------------------------------------------------------------

# 0) 경로
BASE       = OUTPUT_DIR  # 이전 셀에서 사용한 폴더
DATA_PKL   = f"{BASE}/crunch_dataset.pkl"
DATA_CSV   = f"{BASE}/crunch_labels.csv"

import os, json, pickle, warnings, math
import numpy as np, pandas as pd
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score, confusion_matrix
warnings.filterwarnings("ignore")
rng = np.random.default_rng(42)

# 1) 데이터 로드
if os.path.exists(DATA_PKL):
    with open(DATA_PKL, "rb") as f: blob = pickle.load(f)
    X = np.asarray(blob["X"], dtype=np.float32)
    y = np.asarray(blob["y"], dtype=np.int64)
    feature_cols = blob.get("feature_cols", ["hip_angle","sk_ratio","torso_dx_l","torso_dx_r"])
else:
    df = pd.read_csv(DATA_CSV)
    feature_cols = [c for c in ["hip_angle","sk_ratio","torso_dx_l","torso_dx_r"] if c in df.columns]
    X = df[feature_cols].fillna(0.0).to_numpy(np.float32)
    y = df["label"].fillna(0).astype("int64").to_numpy()

print(f"[LOAD] X:{X.shape}, y:{y.shape}, cols={feature_cols}")

# 2) 분할(원본 분포 보존)
X_tr, X_va, y_tr, y_va = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

def count_stats(lbl):
    c = Counter(lbl)
    return {"good_1": int(c.get(1,0)), "bad_0": int(c.get(0,0)), "total": int(len(lbl))}
print("[COUNT TRAIN]", count_stats(y_tr))
print("[COUNT VAL]  ", count_stats(y_va))

# 3) 데이터 증강(정자세=1 대상)
#    - target_ratio: 증강 후 정자세:오자세 비율 (예: 0.8 -> 1:0.8에 가깝게)
target_ratio = 0.8           # 불균형 완화(너무 1:1로 만들면 과적합 위험)
noise_level  = 0.03          # 3% 가우시안 노이즈 (표준편차, 각 피처 스케일 기준)
swap_lr_cols = ("torso_dx_l", "torso_dx_r")  # 좌우반전 시 교환할 컬럼

# 컬럼 인덱스 준비
col_idx = {c:i for i,c in enumerate(feature_cols)}
swap_idx = (col_idx.get(swap_lr_cols[0], None), col_idx.get(swap_lr_cols[1], None))

X_pos = X_tr[y_tr==1]
X_neg = X_tr[y_tr==0]

n_pos, n_neg = len(X_pos), len(X_neg)
# 목표 양성 샘플 수
target_pos = min(int(n_neg * target_ratio), 8 * max(1, n_pos))  # 안정성 위해 과도한 증강 제한
n_to_add = max(0, target_pos - n_pos)
print(f"[AUG] pos:{n_pos} neg:{n_neg} -> add {n_to_add} positives (ratio target ~{target_ratio})")

aug_list = []
for i in range(n_to_add):
    base = X_pos[rng.integers(0, n_pos)].copy()

    # 가우시안 노이즈 (각 피처의 규모에 따른 상대적 노이즈)
    # hip_angle(각도)은 ±(noise_level*180) 정도, ratio들은 ±(noise_level) 정도로 처리
    for j, name in enumerate(feature_cols):
        val = base[j]
        if "angle" in name:
            base[j] = val + rng.normal(0.0, noise_level * 180.0)   # 각도
        else:
            base[j] = val + rng.normal(0.0, noise_level)           # 비율류
    # 물리적 범위 클립
    # 각도: [0, 180], 비율: [-0.5, 2] 정도로 넉넉하게
    for j, name in enumerate(feature_cols):
        if "angle" in name: base[j] = np.clip(base[j], 0.0, 180.0)
        else:               base[j] = np.clip(base[j], -0.5, 2.0)

    # 50% 확률로 좌우반전(대칭 증강): torso_dx_l <-> torso_dx_r 교환
    if swap_idx[0] is not None and swap_idx[1] is not None and rng.random() < 0.5:
        base[swap_idx[0]], base[swap_idx[1]] = base[swap_idx[1]], base[swap_idx[0]]

    aug_list.append(base)

if aug_list:
    X_aug = np.vstack([X_tr, np.vstack(aug_list)])
    y_aug = np.concatenate([y_tr, np.ones(len(aug_list), dtype=np.int64)])
else:
    X_aug, y_aug = X_tr, y_tr

print("[AUG] train shapes:", X_aug.shape, y_aug.shape)
print("[AUG COUNT]", count_stats(y_aug))

# 4) 모델 학습(증강 데이터 사용)
pipe_lr = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=2000, class_weight="balanced"))
])
rf = RandomForestClassifier(
    n_estimators=300,           # 약간 늘림
    max_depth=8,               # 살짝 깊게
    min_samples_leaf=2,
    class_weight="balanced_subsample",
    n_jobs=-1, random_state=42
)

pipe_lr.fit(X_aug, y_aug)
rf.fit(X_aug, y_aug)

def eval_m(m, Xv, yv, name):
    prob = m.predict_proba(Xv)[:,1] if hasattr(m,"predict_proba") else m.decision_function(Xv)
    pred = (prob >= 0.5).astype(int)
    return {
        "name": name,
        "acc": float(accuracy_score(yv, pred)),
        "f1": float(f1_score(yv, pred, zero_division=0)),
        "precision": float(precision_score(yv, pred, zero_division=0)),
        "recall": float(recall_score(yv, pred, zero_division=0)),
        "roc_auc": float(roc_auc_score(yv, prob)),
        "cm": confusion_matrix(yv, pred).tolist()
    }, prob

m_lr, prob_lr = eval_m(pipe_lr, X_va, y_va, "LogReg(v2)")
m_rf, prob_rf = eval_m(rf,      X_va, y_va, "RandomForest(v2)")

best_name, best_model, best_metrics, best_prob = ("LogReg(v2)", pipe_lr, m_lr, prob_lr)
if m_rf["roc_auc"] > m_lr["roc_auc"]:
    best_name, best_model, best_metrics, best_prob = ("RandomForest(v2)", rf, m_rf, prob_rf)

print("[RESULT] LogReg(v2):", m_lr)
print("[RESULT] RandomForest(v2):", m_rf)
print("[SELECT] Best:", best_name)

# 5) 최적 threshold(F1 최대, 정자세 재현율 향상 목적이면 recall-weighted도 고려 가능)
best_thr, best_f1 = 0.5, -1.0
grid = np.linspace(0.05, 0.95, 37)
for t in grid:
    pred_t = (best_prob >= t).astype(int)
    f1 = f1_score(y_va, pred_t, zero_division=0)
    if f1 > best_f1:
        best_f1, best_thr = float(f1), float(t)
print(f"[THR] best_thr={best_thr:.2f}  best_f1={best_f1:.4f}")

# 6) 저장(v2)
with open(f"{BASE}/model_v2.pkl","wb") as f:
    pickle.dump({"model":best_model, "feature_cols":feature_cols, "threshold":best_thr,
                 "meta":{"type":"crunch_binary","selected_model":best_name,"aug":{"target_ratio":target_ratio,"noise":noise_level}}},
                f)
with open(f"{BASE}/feature_cols_v2.json","w",encoding="utf-8") as f:
    json.dump(feature_cols, f, ensure_ascii=False, indent=2)
with open(f"{BASE}/metrics_v2.json","w",encoding="utf-8") as f:
    json.dump({"LogReg_v2":m_lr,"RandomForest_v2":m_rf,"selected":best_name,
               "best_threshold":best_thr,"best_val_f1":best_f1}, f, ensure_ascii=False, indent=2)

pd.DataFrame({
    "y_true": y_va,
    "prob_good": best_prob,
    "pred@0.50": (best_prob>=0.50).astype(int),
    f"pred@{best_thr:.2f}": (best_prob>=best_thr).astype(int)
}).to_csv(f"{BASE}/pred_val_v2.csv", index=False)

with open(f"{BASE}/counts_v2.json","w",encoding="utf-8") as f:
    json.dump({"train_original": count_stats(y_tr),
               "train_augmented": count_stats(y_aug),
               "val": count_stats(y_va)}, f, ensure_ascii=False, indent=2)

print(f"[DONE] v2 saved -> {BASE}")


[LOAD] X:(3237, 4), y:(3237,), cols=['hip_angle', 'sk_ratio', 'torso_dx_l', 'torso_dx_r']
[COUNT TRAIN] {'good_1': 34, 'bad_0': 2555, 'total': 2589}
[COUNT VAL]   {'good_1': 8, 'bad_0': 640, 'total': 648}
[AUG] pos:34 neg:2555 -> add 238 positives (ratio target ~0.8)
[AUG] train shapes: (2827, 4) (2827,)
[AUG COUNT] {'good_1': 272, 'bad_0': 2555, 'total': 2827}
[RESULT] LogReg(v2): {'name': 'LogReg(v2)', 'acc': 0.9814814814814815, 'f1': 0.5714285714285714, 'precision': 0.4, 'recall': 1.0, 'roc_auc': 0.9968750000000001, 'cm': [[628, 12], [0, 8]]}
[RESULT] RandomForest(v2): {'name': 'RandomForest(v2)', 'acc': 0.9953703703703703, 'f1': 0.8235294117647058, 'precision': 0.7777777777777778, 'recall': 0.875, 'roc_auc': 0.99765625, 'cm': [[638, 2], [1, 7]]}
[SELECT] Best: RandomForest(v2)
[THR] best_thr=0.30  best_f1=0.8235
[DONE] v2 saved -> /content/drive/MyDrive/crunch/outputs


In [7]:
# === Crunch 재학습(강화판, 한 셀 완결) ===============================
# - 양성(정자세) 1:1 근처까지 오버샘플 + 가벼운 피처 증강
# - F2(Recall 가중) + Precision 하한으로 임계값 선택
# - 저장: model_v2b.pkl, feature_cols_v2b.json, metrics_v2b.json, counts_v2b.json, pred_val_v2b.csv
# ===================================================================

# 0) 경로 (필요 시만 수정)
BASE       = OUTPUT_DIR  # 이전 라벨링/학습 셀에서 사용한 경로 재사용
DATA_PKL   = f"{BASE}/crunch_dataset.pkl"
DATA_CSV   = f"{BASE}/crunch_labels.csv"

import os, json, pickle, warnings
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import (precision_score, recall_score, fbeta_score, roc_auc_score,
                             accuracy_score, confusion_matrix, f1_score)

rng = np.random.default_rng(123)

# 1) 데이터 로드
if os.path.exists(DATA_PKL):
    with open(DATA_PKL, "rb") as f: blob = pickle.load(f)
    X = np.asarray(blob["X"], np.float32); y = np.asarray(blob["y"], np.int64)
    cols = blob.get("feature_cols", ["hip_angle","sk_ratio","torso_dx_l","torso_dx_r"])
else:
    df = pd.read_csv(DATA_CSV)
    cols = [c for c in ["hip_angle","sk_ratio","torso_dx_l","torso_dx_r"] if c in df.columns]
    X = df[cols].fillna(0.0).to_numpy(np.float32); y = df["label"].astype("int64").to_numpy()

def cnt(lbl):
    c = Counter(lbl); return {"good_1": int(c.get(1,0)), "bad_0": int(c.get(0,0)), "total": len(lbl)}

print(f"[LOAD] X:{X.shape}, y:{y.shape}, cols={cols}")

# 2) 분할(원본 분포 유지)
X_tr, X_va, y_tr, y_va = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
print("[COUNT TRAIN orig]", cnt(y_tr))
print("[COUNT VAL   orig]", cnt(y_va))

# 3) 오버샘플 + 증강 (정자세=1만)
target_ratio = 1.0   # 음성(0) 대비 양성(1) 비율 목표 ≈ 1:1
noise_level  = 0.02  # 각도: ±(noise*180), 비율: ±noise
max_cap_mult = 20    # 원양성 수 대비 최대 20배까지

pos = X_tr[y_tr==1]; neg = X_tr[y_tr==0]
n_pos, n_neg = len(pos), len(neg)
target_pos = min(int(n_neg * target_ratio), max_cap_mult * max(1, n_pos))
n_add = max(0, target_pos - n_pos)
print(f"[AUG PLAN] pos:{n_pos}, neg:{n_neg} -> add {n_add} positives")

aug = []
for _ in range(n_add):
    b = pos[rng.integers(0, n_pos)].copy()
    for j, name in enumerate(cols):
        if "angle" in name:
            b[j] += rng.normal(0.0, noise_level * 180.0)
            b[j]  = np.clip(b[j], 0.0, 180.0)
        else:
            b[j] += rng.normal(0.0, noise_level)
            b[j]  = np.clip(b[j], -0.5, 2.0)
    # 좌우반전 50%: torso_dx_l <-> torso_dx_r
    if "torso_dx_l" in cols and "torso_dx_r" in cols and rng.random() < 0.5:
        li, ri = cols.index("torso_dx_l"), cols.index("torso_dx_r")
        b[li], b[ri] = b[ri], b[li]
    aug.append(b)

X_aug = np.vstack([X_tr, np.vstack(aug)]) if aug else X_tr
y_aug = np.concatenate([y_tr, np.ones(len(aug), np.int64)]) if aug else y_tr
print("[COUNT TRAIN aug ]", cnt(y_aug))

# 4) 모델 학습
pipe_lr = Pipeline([("scaler", StandardScaler()),
                    ("clf", LogisticRegression(max_iter=3000, class_weight="balanced"))])
rf = RandomForestClassifier(n_estimators=400, max_depth=10, min_samples_leaf=2,
                            class_weight="balanced_subsample", n_jobs=-1, random_state=42)
pipe_lr.fit(X_aug, y_aug); rf.fit(X_aug, y_aug)

# 5) 검증 + 임계값 선택(F2 우선, Precision 하한)
def eval_pick(model, Xv, yv, name, beta=2.0, prec_floor=0.80):
    prob = model.predict_proba(Xv)[:,1]
    grid = np.linspace(0.05, 0.95, 181)
    best = {"thr":0.5, "f2":-1, "p":0, "r":0, "f1":0}
    for t in grid:
        pred = (prob >= t).astype(int)
        p = precision_score(yv, pred, zero_division=0)
        r = recall_score(yv, pred, zero_division=0)
        f2 = fbeta_score(yv, pred, beta=beta, zero_division=0)
        if p >= prec_floor and f2 > best["f2"]:
            best = {"thr":float(t), "f2":float(f2), "p":float(p), "r":float(r), "f1":float(f1_score(yv,pred,zero_division=0))}
    # 하한 만족 임계값이 없으면 F2 최대만 채택
    if best["f2"] < 0:
        for t in grid:
            pred = (prob >= t).astype(int)
            f2 = fbeta_score(yv, pred, beta=beta, zero_division=0)
            p = precision_score(yv, pred, zero_division=0)
            r = recall_score(yv, pred, zero_division=0)
            f1 = f1_score(yv, pred, zero_division=0)
            if f2 > best["f2"]:
                best = {"thr":float(t), "f2":float(f2), "p":float(p), "r":float(r), "f1":float(f1)}
    metrics = {
        "name": name,
        "precision": best["p"], "recall": best["r"],
        "f1@best": best["f1"], "f2@best": best["f2"],
        "roc_auc": float(roc_auc_score(yv, prob)),
        "acc@best": float(accuracy_score(yv, (prob >= best['thr']).astype(int))),
        "cm@best": confusion_matrix(yv, (prob >= best['thr']).astype(int)).tolist()
    }
    return metrics, prob, best

m_lr, prob_lr, best_lr = eval_pick(pipe_lr, X_va, y_va, "LogReg(v2b)")
m_rf, prob_rf, best_rf = eval_pick(rf,      X_va, y_va, "RandomForest(v2b)")

# F2 기준 선택
use_rf = m_rf["f2@best"] >= m_lr["f2@best"]
metrics, model, prob, best_sel = (m_rf, rf, prob_rf, best_rf) if use_rf else (m_lr, pipe_lr, prob_lr, best_lr)

chosen_thr = best_sel.get("best_thr", best_sel.get("thr"))  # <- 키 안전 처리

print("[RESULT] LogReg(v2b):", m_lr)
print("[RESULT] RandomForest(v2b):", m_rf)
print("[SELECT] %s | thr=%.2f | P=%.3f R=%.3f F1=%.3f F2=%.3f | AUC=%.3f" %
      (metrics["name"], chosen_thr, metrics["precision"], metrics["recall"],
       metrics["f1@best"], metrics["f2@best"], metrics["roc_auc"]))

# 6) 저장
with open(f"{BASE}/model_v2b.pkl","wb") as f:
    pickle.dump({
        "model": model,
        "feature_cols": cols,
        "threshold": float(chosen_thr),
        "meta": {
            "type":"crunch_binary",
            "selected": metrics["name"],
            "criterion":"F2_with_precision_floor",
            "prec_floor": 0.80
        }
    }, f)

with open(f"{BASE}/feature_cols_v2b.json","w",encoding="utf-8") as f:
    json.dump(cols, f, ensure_ascii=False, indent=2)

with open(f"{BASE}/metrics_v2b.json","w",encoding="utf-8") as f:
    json.dump({"LogReg_v2b": m_lr, "RandomForest_v2b": m_rf, "selected": metrics["name"]}, f, ensure_ascii=False, indent=2)

# 검증 예측 저장
pd.DataFrame({
    "y_true": y_va,
    "prob_good": prob,
    f"pred@{chosen_thr:.2f}": (prob >= chosen_thr).astype(int)
}).to_csv(f"{BASE}/pred_val_v2b.csv", index=False)

# 증강/분포 기록
with open(f"{BASE}/counts_v2b.json","w",encoding="utf-8") as f:
    json.dump({"train_original": cnt(y_tr),
               "train_augmented": cnt(y_aug),
               "val": cnt(y_va)}, f, ensure_ascii=False, indent=2)

print(f"[DONE] v2b saved -> {BASE}")


[LOAD] X:(3237, 4), y:(3237,), cols=['hip_angle', 'sk_ratio', 'torso_dx_l', 'torso_dx_r']
[COUNT TRAIN orig] {'good_1': 34, 'bad_0': 2555, 'total': 2589}
[COUNT VAL   orig] {'good_1': 8, 'bad_0': 640, 'total': 648}
[AUG PLAN] pos:34, neg:2555 -> add 646 positives
[COUNT TRAIN aug ] {'good_1': 680, 'bad_0': 2555, 'total': 3235}
[RESULT] LogReg(v2b): {'name': 'LogReg(v2b)', 'precision': 0.7777777777777778, 'recall': 0.875, 'f1@best': 0.8235294117647058, 'f2@best': 0.8536585365853658, 'roc_auc': 0.9958984375000001, 'acc@best': 0.9953703703703703, 'cm@best': [[638, 2], [1, 7]]}
[RESULT] RandomForest(v2b): {'name': 'RandomForest(v2b)', 'precision': 0.8333333333333334, 'recall': 0.625, 'f1@best': 0.7142857142857143, 'f2@best': 0.6578947368421053, 'roc_auc': 0.9978515625000001, 'acc@best': 0.9938271604938271, 'cm@best': [[639, 1], [3, 5]]}
[SELECT] LogReg(v2b) | thr=0.86 | P=0.778 R=0.875 F1=0.824 F2=0.854 | AUC=0.996
[DONE] v2b saved -> /content/drive/MyDrive/crunch/outputs
